# 키트별 정답 탐 음높이 분포 — GPU 서버 실행 (ai_2_to_4 11/06)

탐 세 종류가 음높이로 뚜렷하게 나뉘는지 키트가 많은 데이터로 확인한다. 로컬 GMD(전자 드럼 키트 하나)에서는 깔끔하게 나뉘었다.

- 데이터: E-GMD test·validation 어쿠스틱 26킷(1,170개·3,042개), STAR test·validation(31곡·119곡, 샘플 키트 20종)
- 측정: 정답 드럼 트랙에서 정답 탐 타격마다 50~400Hz 스펙트럼 최대 피크(서비스 규칙과 같은 계산, 120ms 구간). 앞 30ms~뒤 120ms 안에 하이햇 말고 다른 타격이 있는 탐은 뺌
- GPU를 쓰지 않는다. CPU로 10~20분

**준비물.** 이 파일만 JupyterHub `ssaplash/notebooks/`에 올린다. 코드 zip은 올리지 않는다. 서버 `~/ssaplash/S15P31A307`은 운영 워커 코드라 덮어쓰지 않고, 그 `.venv`와 데이터 로더를 읽기만 한다. 측정 스크립트는 2번 셀이 `~/ssaplash/tom_pitch/`에 새로 쓴다.

**순서.** 1 설정 → 2 스크립트 쓰기 → 3 데이터 점검 → 4 곡 3개씩 시험 → 5 전체 시작(백그라운드, 한 번만) → 6 진행 확인(여러 번) → 7 결과 보기·묶기.

**끊기면.** 5번은 nohup이라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 1번만 다시 치고 6번 또는 7번부터 이어 간다.

In [ ]:
# 1. 설정
import getpass
import os

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"   # nohup 로그가 바로바로 써지게
os.environ["MPLBACKEND"] = "Agg"
# 곡 읽기 작업 하나가 코어 1개만 쓰게 함. 안 하면 numpy가 작업마다 서버 코어를 다 쓰려 해서 서로 뺏음
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"           # 읽기만 함
VPY = f"{AI}/.venv/bin/python"
TOOL = f"{WORK}/tom_pitch"
SCRIPT = f"{TOOL}/tom_pitch_kits.py"
OUT = f"{WORK}/results/tom-pitch-kits-1106"
TEST_OUT = f"{TOOL}/test_out"
LOG = f"{TOOL}/tom_pitch_kits_1106.log"
ME = getpass.getuser()
DS = "configs/bench/datasets"
SETS = [f"{DS}/egmd_test_acoustic.yaml", f"{DS}/egmd_validation_acoustic.yaml",
        f"{DS}/star_drums_test.yaml", f"{DS}/star_drums_validation.yaml"]
WORKERS = 32
os.makedirs(TOOL, exist_ok=True)
os.makedirs(f"{WORK}/results", exist_ok=True)
%cd $AI
print("CPU 코어", os.cpu_count(), "· 작업", WORKERS)
!ls -d $VPY data/e-gmd/e-gmd-v1.0.0 data/star_drums/full/data
!git -C $AI/.. log --oneline -1
# 확인: 코어 수(256 안팎)와 경로 3개가 찍히고, 커밋 줄 하나(6023e0e 등 운영 워커 커밋).
#       경로가 없다고 나오면 그 줄을 적어 두고 멈춤 (데이터 링크가 없어졌을 수 있음)

In [ ]:
# 2. 측정 스크립트 쓰기 (여러 번 쳐도 됨. 같은 내용으로 덮어씀)
SCRIPT_TEXT = r'''
"""키트별 정답 탐 음높이 분포 (ai_2_to_4 11/06, 탐 그룹이 키트가 여럿일 때도 나뉘는지 확인).

정답 드럼 트랙에서 정답 탐 타격마다 50~400Hz 스펙트럼 최대 피크(서비스 규칙 estimate_f0 그대로, 120ms 구간)를 잼.
앞 30ms~뒤 120ms 안에 하이햇·페달 말고 다른 타격이 있는 탐은 뺌(겹친 소리로 음높이가 흐려짐).
S15P31A307 코드는 읽기만 함(데이터 로더·estimate_f0 import).

  python tom_pitch_kits.py --dataset configs/bench/datasets/egmd_test_acoustic.yaml ... --out <폴더> --workers 32
"""

from __future__ import annotations

import argparse
import csv
import os
from collections import defaultdict
from multiprocessing import Pool
from pathlib import Path

import numpy as np

PADS = ("FLOOR_TOM", "MID_TOM", "SMALL_TOM")
NAME = {"FLOOR_TOM": "Floor", "MID_TOM": "Mid", "SMALL_TOM": "Small"}
COLOR = {"FLOOR_TOM": "#2a78d6", "MID_TOM": "#eb6834", "SMALL_TOM": "#1baf7a"}
SKIP_NEAR = ("HIHAT", "HIHAT_PEDAL")
PRE, WIN = 0.03, 0.12


def st(hz):
    return 12 * np.log2(np.asarray(hz, dtype=float) / 55.0)


_DS = {}


def _dataset(path: str):
    from ssaplash_ai.bench.datasets import load_dataset

    if path not in _DS:
        _DS[path] = load_dataset(path)
    return _DS[path]


def measure(job: tuple[str, str]) -> tuple[list[tuple], int, str]:
    """(데이터셋 yaml, 곡) → [(데이터셋, 키트, 곡, 패드, Hz)], 정답 탐 수, 오류"""
    from ssaplash_ai.bench.transcribers.adtof_drumsep_transcriber import estimate_f0

    path, song = job
    try:
        ds = _dataset(path)
        item = ds.load(song)
        hits = item.hits_true
        toms = hits[np.isin(hits["pad"], PADS)]
        if not toms.size:
            return [], 0, ""
        kit = ds.kit_of(song) if hasattr(ds, "kit_of") else "-"
        audio = item.drums_true
        mono = audio.mean(axis=0) if audio.ndim == 2 else audio
        sr = item.sample_rate
        others = np.sort(hits["time_sec"][~np.isin(hits["pad"], SKIP_NEAR)])
        out = []
        for t, p in zip(toms["time_sec"], toms["pad"], strict=True):
            near = others[(others > t - PRE) & (others < t + WIN)]
            if np.any(np.abs(near - t) > 1e-4):
                continue
            a = int(round(t * sr))
            f0 = estimate_f0(mono[a : a + int(WIN * sr)].astype(np.float32), sr, 50.0, 400.0)
            if np.isfinite(f0):
                out.append((ds.name, kit, song, str(p), float(f0)))
        return out, int(toms.size), ""
    except Exception as e:  # noqa: BLE001 - 곡 하나 실패는 기록하고 넘어감
        return [], 0, f"{song}: {type(e).__name__}: {e}"


def best_split_acc(f: np.ndarray, lab: np.ndarray) -> float:
    """정답 이름을 알 때 음높이 낮은 순으로 경계를 그어 맞힐 수 있는 최대 비율 (그룹 2~3개)"""
    order = np.argsort(f)
    lab = lab[order]
    uniq = sorted(set(lab.tolist()))
    n = lab.size
    # 경계 c에서 왼쪽은 uniq[0], 오른쪽은 uniq[1]... 누적합으로 계산
    cum = {u: np.concatenate([[0], np.cumsum(lab == u)]) for u in uniq}
    if len(uniq) == 2:
        a, b = uniq
        return max((cum[a][c] + (cum[b][n] - cum[b][c])) / n for c in range(n + 1))
    a, b, c3 = uniq
    best = 0.0
    for c1 in range(n + 1):
        left = cum[a][c1]
        mid = cum[b][c1:] - cum[b][c1]
        right = cum[c3][n] - cum[c3][c1:]
        best = max(best, float(np.max(left + mid + right)) / n)
    return best


def kit_summary(rows: list[tuple]) -> list[dict]:
    by_kit = defaultdict(list)
    for r in rows:
        by_kit[(r[0], r[1])].append(r)
    out = []
    for (dsname, kit), rs in sorted(by_kit.items()):
        d = {"dataset": dsname, "kit": kit, "hits": len(rs)}
        for p in PADS:
            a = np.array([r[4] for r in rs if r[3] == p])
            d[f"n_{NAME[p]}"] = a.size
            d[f"med_{NAME[p]}"] = round(float(np.median(a)), 1) if a.size else ""
            d[f"iqr_{NAME[p]}"] = f"{np.percentile(a, 25):.0f}-{np.percentile(a, 75):.0f}" if a.size else ""
        songs = defaultdict(list)
        for r in rs:
            songs[r[2]].append((r[3], r[4]))
        accs, gaps = [], []
        for v in songs.values():
            kinds = {p for p, _ in v}
            if len(kinds) < 2 or len(v) < 10:
                continue
            f = st([x for _, x in v])
            lab = np.array([PADS.index(p) for p, _ in v])
            accs.append(best_split_acc(f, lab))
            meds = [np.median(f[lab == u]) for u in sorted(set(lab.tolist()))]
            gaps += list(np.diff(meds))
        d["songs_2plus"] = len(accs)
        d["split_acc_median"] = round(float(np.median(accs)), 3) if accs else ""
        d["songs_acc_ge_095"] = round(float(np.mean(np.array(accs) >= 0.95)), 3) if accs else ""
        d["gap_st_median"] = round(float(np.median(gaps)), 2) if gaps else ""
        d["gap_lt3_frac"] = round(float(np.mean(np.array(gaps) < 3)), 3) if gaps else ""
        d["gap_neg_frac"] = round(float(np.mean(np.array(gaps) < 0)), 3) if gaps else ""
        out.append(d)
    return out


def plot(rows: list[tuple], dsname: str, out_png: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    kits = sorted({r[1] for r in rows if r[0] == dsname})
    if not kits:
        return
    ncol = 5
    nrow = (len(kits) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.3 * nrow), sharex=True, squeeze=False)
    bins = np.arange(st(50), st(400) + 0.25, 0.25)
    ticks = [50, 70, 100, 140, 200, 280, 400]
    for ax, kit in zip(axes.flat, kits, strict=False):
        for p in PADS:
            a = st([r[4] for r in rows if r[0] == dsname and r[1] == kit and r[3] == p])
            if a.size:
                ax.hist(a, bins=bins, histtype="step", linewidth=1.5, color=COLOR[p], label=NAME[p])
        ax.set_title(kit, fontsize=8)
        ax.tick_params(labelsize=6)
        ax.set_xticks(st(ticks), [str(t) for t in ticks])
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    for ax in list(axes.flat)[len(kits) :]:
        ax.axis("off")
    axes.flat[0].legend(fontsize=7, frameon=False)
    fig.suptitle(f"{dsname}: true tom pitch per kit (Hz, semitone-spaced axis)", fontsize=10)
    fig.tight_layout()
    fig.savefig(out_png, dpi=120)
    plt.close(fig)


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--dataset", nargs="+", required=True)
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) // 2))
    ap.add_argument("--limit", type=int, default=None, help="데이터셋마다 앞에서 이만큼만 (확인용)")
    args = ap.parse_args()
    args.out.mkdir(parents=True, exist_ok=True)
    jobs = []
    for path in args.dataset:
        songs = _dataset(path).song_ids()
        if args.limit:
            songs = songs[: args.limit]
        print(f"[tom-pitch] {path}: 곡 {len(songs)}개", flush=True)
        jobs += [(path, s) for s in songs]
    rows, n_true, errors = [], 0, []
    with Pool(args.workers) as pool:
        for i, (r, n, err) in enumerate(pool.imap_unordered(measure, jobs, chunksize=4), 1):
            rows += r
            n_true += n
            if err:
                errors.append(err)
            if i % 200 == 0 or i == len(jobs):
                print(f"[tom-pitch] {i}/{len(jobs)}곡, 쓴 탐 {len(rows)}개, 오류 {len(errors)}", flush=True)
    with open(args.out / "hits.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["dataset", "kit", "song", "pad", "hz"])
        w.writerows(rows)
    summ = kit_summary(rows)
    with open(args.out / "kits.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(summ[0].keys()))
        w.writeheader()
        w.writerows(summ)
    for dsname in sorted({r[0] for r in rows}):
        plot(rows, dsname, args.out / f"{dsname}_kits.png")
    (args.out / "errors.txt").write_text("\n".join(errors), encoding="utf-8")
    print(f"[tom-pitch] 정답 탐 {n_true}개 중 안 겹친 {len(rows)}개 사용, 오류 곡 {len(errors)}개", flush=True)
    print(f"[tom-pitch] 저장 {args.out}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''
with open(SCRIPT, "w", encoding="utf-8") as f:
    f.write(SCRIPT_TEXT.lstrip("\n"))
!wc -l $SCRIPT
# 확인: '숫자 .../tom_pitch/tom_pitch_kits.py' 한 줄 (205줄)

In [ ]:
# 3. 데이터 점검 (여러 번 쳐도 됨)
!$AI/.venv/bin/ssaplash-bench check-data {" ".join(SETS)}
# 확인: egmd_test_acoustic 1,170개, egmd_validation_acoustic 3,042개, star_drums_test 31곡, star_drums_validation 119곡,
#       모두 누락 0. 누락이 있으면 그 데이터 이름과 누락 수를 적어 두고 멈춤

In [ ]:
# 4. 곡 3개씩 시험 (1분 안쪽)
!$VPY $SCRIPT --dataset {" ".join(SETS)} --limit 3 --workers 4 --out $TEST_OUT
!head -3 $TEST_OUT/kits.csv; cat $TEST_OUT/errors.txt
# 확인: '[tom-pitch] ...: 곡 1170개' 같은 줄 4개, '12/12곡, 쓴 탐 N개, 오류 0', kits.csv 머리줄과 키트 줄,
#       errors.txt는 비어 있음. Traceback이 나오면 마지막 10줄을 그대로 적어 둠

In [ ]:
# 5. 전체 시작 (백그라운드, 한 번만)
import subprocess

assert not os.path.exists(LOG), "이미 시작했음. 6번 셀로 진행을 확인함"
cmd = [VPY, SCRIPT, "--dataset", *SETS, "--workers", str(WORKERS), "--out", OUT]
proc = subprocess.Popen(["nohup", "nice", "-n", "19", *cmd], cwd=AI, stdout=open(LOG, "w"),
                        stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
print("시작. pid", proc.pid, "로그", LOG)
# 확인: '시작. pid 숫자' 한 줄. 바로 6번으로

In [ ]:
# 6. 진행 확인 (여러 번 쳐도 됨)
!tail -3 $LOG
!ps -u $ME -o pid,etime,args | grep -E "[t]om_pitch_kits" | head -1 | cut -c1-120 || true
# 확인: 로그 끝이 '[tom-pitch] 200/4362곡, 쓴 탐 N개, 오류 0'처럼 늘어남. 끝나면 '[tom-pitch] 저장 ...' 줄이 나오고 ps 줄이 없어짐.
#       오류 수가 0보다 크면 끝난 뒤 7번에서 errors.txt를 봄

In [ ]:
# 7. 결과 보기·묶기 (끝난 뒤). 결과 zip을 내려받아 로컬 AI/data/results/tom-pitch-2026-11-06/ 에 둠
# Jupyter 기본 커널에는 pandas가 없어 표준 csv 모듈로 읽음
import csv
from IPython.display import Image, display

COLS = ["dataset", "kit", "hits", "med_Floor", "med_Mid", "med_Small", "songs_2plus", "split_acc_median",
        "songs_acc_ge_095", "gap_st_median", "gap_lt3_frac", "gap_neg_frac"]
with open(f"{OUT}/kits.csv", encoding="utf-8") as f:
    rows = [[r[c] for c in COLS] for r in csv.DictReader(f)]
width = [max(len(str(x)) for x in col) for col in zip(COLS, *rows)]
for line in [COLS, *rows]:
    print("  ".join(str(x).ljust(w) for x, w in zip(line, width)))
for name in ("egmd_test_acoustic", "egmd_validation_acoustic", "star_drums_test", "star_drums_validation"):
    p = f"{OUT}/{name}_kits.png"
    if os.path.exists(p):
        display(Image(p))
!wc -l $OUT/errors.txt
!rm -f $WORK/results/tom_pitch_kits_1106.zip && cd $WORK/results && $VPY -m zipfile -c tom_pitch_kits_1106.zip tom-pitch-kits-1106 $LOG && ls -la tom_pitch_kits_1106.zip
# 확인: E-GMD 26킷 × 2, STAR 키트 줄이 표로 찍히고 그림 4장이 나옴. errors.txt 줄 수 0.
#       zip은 수 MB. 표가 비었거나 그림이 안 나오면 로그 끝 10줄을 적어 둠